#  Unittest
This notebook is used to validate that all core design patterns in the `financial_system` project are working as expected.

## 1. Setup and Imports

First, we import all necessary modules. Please ensure you run this notebook from the project's root directory.

In [8]:
from unittest.mock import Mock
import sys
import os

# Assuming this notebook is in tests/, we need to add the root directory to the path
# This allows imports like 'from models import ...' to work
if os.path.basename(os.getcwd()) == 'tests':
    os.chdir('..')
    sys.path.append(os.getcwd())
    print(f"Changed directory to: {os.getcwd()}")

from models import Stock, Bond, Instrument
from patterns.factory import InstrumentFactory
from patterns.singleton import Config
from analytics import VolatilityDecorator, BetaDecorator, DrawdownDecorator
from patterns.observer import SignalPublisher, Observer
from patterns.command import ExecuteOrderCommand, CommandInvoker
from patterns.strategy import MeanReversionStrategy
from data_loader import MarketDataPoint

print(" All modules imported successfully!")

 All modules imported successfully!


---

##  Test 1: Factory Pattern

**Validation:** `InstrumentFactory` can create the correct `Instrument` subclass based on dictionary data.

In [9]:
print("--- Test 1: Factory Pattern ---")
stock_data = {'type': 'Stock', 'symbol': 'TEST'}
bond_data = {'type': 'Bond', 'symbol': 'BND'}

stock_obj = InstrumentFactory.create_instrument(stock_data)
bond_obj = InstrumentFactory.create_instrument(bond_data)

# 1. Test type creation
assert isinstance(stock_obj, Stock)
assert isinstance(bond_obj, Bond)
print("  -> Type check PASSED.")

# # 2. Test attribute assignment
print(stock_obj.symbol)
assert stock_obj.symbol == 'TEST'
assert stock_obj.type == 'Stock' # Name should default to symbol
assert bond_obj.type == 'Bond'
print("  -> Attribute check PASSED.")

# 3. Test error handling for unknown type
try:
    InstrumentFactory.create_instrument({'type': 'Future', 'symbol': 'FUT'})
    assert False, "Did not raise ValueError as expected"
except ValueError as e:
    assert "Unknown instrument type" in str(e)
    print("  -> Error handling check PASSED.")

print("\n Test 1 PASSED!")

--- Test 1: Factory Pattern ---
  -> Type check PASSED.
TEST
  -> Attribute check PASSED.
  -> Error handling check PASSED.

 Test 1 PASSED!


---

##  Test 2: Singleton Pattern

**Validation:** The `Config` class always returns the same instance when called multiple times.

In [10]:
print("--- Test 2: Singleton Pattern ---")

config1 = Config()
config2 = Config()

# 1. Test that both variables point to the same object
assert config1 is config2
print("  -> Instance identity check (config1 is config2) PASSED.")

# 2. Test shared state
config1._settings = {"test_key": "test_value"} # Simulate loading
assert config2.get("test_key") == "test_value"
print("  -> Shared state check PASSED.")

print("\n Test 2 PASSED!")

--- Test 2: Singleton Pattern ---
  -> Instance identity check (config1 is config2) PASSED.
  -> Shared state check PASSED.

 Test 2 PASSED!


---

##  Test 3: Decorator Pattern

**Validation:** Decorators can add new analytics to `get_metrics()` without modifying the base class.

In [11]:
print("--- Test 3: Decorator Pattern ---")

base_stock = Stock(symbol='DECOR', name='Decorator Stock')
print(f"  Base metrics: {base_stock.get_metrics()}")

# Demonstrate stacking
decorated_stock = DrawdownDecorator(BetaDecorator(VolatilityDecorator(base_stock)))
metrics = decorated_stock.get_metrics()
print(f"  Decorated metrics: {metrics}")

# 1. Validate all metrics are present
assert 'symbol' in metrics
assert 'volatility' in metrics
assert 'beta' in metrics
assert 'max_drawdown' in metrics

# 2. Check preset values
assert metrics['beta'] == 1.2

print("\n Test 3 PASSED!")

--- Test 3: Decorator Pattern ---
  Base metrics: {'symbol': 'DECOR'}
  Decorated metrics: {'symbol': 'DECOR', 'volatility': 0.25, 'beta': 1.2, 'max_drawdown': 0.15}

 Test 3 PASSED!


---

##  Test 4: Observer & Command Patterns

**Validation (Observer):** `SignalPublisher.notify()` triggers the `update()` method of all registered observers.

**Validation (Command):** `CommandInvoker` correctly executes `execute()` and `undo()` and manages command history.

In [12]:
print("--- Test 4a: Observer Pattern ---")

publisher = SignalPublisher()
mock_observer_1 = Mock(spec=Observer)
mock_observer_2 = Mock(spec=Observer)

publisher.attach(mock_observer_1)
publisher.attach(mock_observer_2)

test_signal = {'action': 'BUY', 'symbol': 'OBSV', 'price': 100}
publisher.notify(test_signal)

# 1. Validate both observers were called with the correct signal
mock_observer_1.update.assert_called_once_with(test_signal)
mock_observer_2.update.assert_called_once_with(test_signal)

print("  -> Observer notification check PASSED.")
print("\n--- Test 4b: Command Pattern ---")

invoker = CommandInvoker()
mock_portfolio = Mock()
command = ExecuteOrderCommand(mock_portfolio, test_signal)

# 2. Test execute: We care that portfolio.transact is called
invoker.execute_command(command)
assert len(invoker._history) == 1
# Validate the real transaction logic was triggered with correct args
mock_portfolio.transact.assert_called_once_with('OBSV', 10, 100)
print("  -> Command execution check PASSED.")

# 3. Test undo
invoker.undo_last_command()
assert len(invoker._history) == 0
# Validate transact was called a second time (since undo is also a transact)
assert mock_portfolio.transact.call_count == 2
print("  -> Command undo check PASSED.")

print("\n Test 4 PASSED!")

--- Test 4a: Observer Pattern ---
  -> Observer notification check PASSED.

--- Test 4b: Command Pattern ---
[COMMAND] Executing BUY for 10 shares of OBSV...
  -> Trade Executed.
  -> Command execution check PASSED.
[COMMAND] Undoing BUY for OBSV...
  -> Undo Executed.
  -> Command undo check PASSED.

 Test 4 PASSED!


---

##  Test 5: Strategy Pattern

**Validation:** `MeanReversionStrategy` generates correct signals (with no look-ahead bias) and handles symbols independently.

In [13]:
print("--- Test 5: Strategy Pattern ---")

# 1. Use a small window for easy testing
strategy = MeanReversionStrategy(lookback_window=3, threshold=0.05)

# History: 100, 105, 110 -> Mean is 105
# BUY signal triggers at: < 99.75 (105 * (1-0.05))
# SELL signal triggers at: > 110.25 (105 * (1+0.05))
ticks = [
    MarketDataPoint('STRAT', 100, 'T1'), # Fill 1
    MarketDataPoint('STRAT', 105, 'T2'), # Fill 2
    MarketDataPoint('STRAT', 110, 'T3'), # Fill 3 (Window full, history [100, 105])
    MarketDataPoint('STRAT', 99,  'T4'), # Should trigger BUY (History [100, 105, 110])
    MarketDataPoint('STRAT', 111, 'T5')  # Should trigger SELL (History [105, 110, 99])
]

# T1, T2: Fill the deque
signals_t1 = strategy.generate_signals(ticks[0])
signals_t2 = strategy.generate_signals(ticks[1])
assert len(signals_t1) == 0 and len(signals_t2) == 0
print("  -> Filling history... (T1, T2)")

# T3: Window is full for the first time. History [100, 105]. Mean=102.5. T3 price 110, no trigger.
signals_t3 = strategy.generate_signals(ticks[2])
assert len(signals_t3) == 0
print("  -> Window full... (T3)")

# T4: History is [100, 105, 110]. Mean=105. T4 price 99.  99 < 99.75, triggers BUY.
signals_t4 = strategy.generate_signals(ticks[3])
assert len(signals_t4) == 1
assert signals_t4[0]['action'] == 'BUY'
print("  -> BUY signal check PASSED. (T4)")

# T5: History is [105, 110, 99]. Mean=104.67. T5 price 111. 111 > 110.25, triggers SELL.
signals_t5 = strategy.generate_signals(ticks[4])
assert len(signals_t5) == 1
assert signals_t5[0]['action'] == 'SELL'
print("  -> SELL signal check PASSED. (T5)")

# 2. Test that symbols are handled independently
strategy.generate_signals(MarketDataPoint('OTHER', 1000, 'T6'))
strategy.generate_signals(MarketDataPoint('OTHER', 1000, 'T7'))
strategy.generate_signals(MarketDataPoint('OTHER', 1000, 'T8'))
signals_t9 = strategy.generate_signals(MarketDataPoint('STRAT', 90, 'T9')) # History [110, 99, 111], Mean=106.67
assert signals_t9[0]['action'] == 'BUY' # 90 < 106.67 * 0.95
print("  -> Independent symbol check PASSED.")

print("\n Test 5 PASSED!")

--- Test 5: Strategy Pattern ---
Initialized MeanReversionStrategy: window=3, threshold=0.05
  -> Filling history... (T1, T2)
  -> Window full... (T3)
  -> BUY signal check PASSED. (T4)
  -> SELL signal check PASSED. (T5)
  -> Independent symbol check PASSED.

 Test 5 PASSED!


---

##  Conclusion

All 5 core validation points have passed. The system components are working together as expected.